# Codeletion (1p/19q): IPD Brain secondary check

Secondary, caveated finding, not part of the primary RQ2-generalisation
claim (that's `codel_selective_prediction_bench.ipynb`, TCGA/EBRAINS).

The codeletion label here is a Subtype-based proxy (`OLIGODENDROGLIOMA` vs
`ASTROCYTOMA`, IDH-mutant only), not a genomic assay: IPD Brain has no
1p/19q/FISH data at all. Corroborated by ATRX IHC status independently:
ATRX and Subtype-derived codel status agree in 127/138 (92%) patients,
matching the expected biological pattern (ATRX loss with astrocytoma, ATRX
retained with oligodendroglioma, near-mutually-exclusive in practice,
which is why ATRX is a clinically-used surrogate for 1p/19q status).
Reported in two tiers: the ATRX-concordant "confident" tier (n=127) as the
headline secondary result, and the full 138 (including the 11
ATRX-discordant cases) as a robustness check.

Uses the EBRAINS-trained codel ABMIL/UQ-family models externally on IPD
Brain, no refitting. All UQ-family scoring (MC-dropout, deep-ensemble,
Laplace) was already run; only OOD needed a fresh score (flow
`topk`/`mean` reused from the main IDH bench's IPD scoring where
classifier-independent, `attention` freshly computed against the codel
classifier).

In [ ]:
from __future__ import annotations

from pathlib import Path
from itertools import combinations
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, kruskal, fisher_exact, norm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    f1_score, brier_score_loss, confusion_matrix, roc_curve,
)

from IPython.display import display

warnings.filterwarnings("ignore")   # bootstrap loops on tiny subgroups trip harmless
                                     # RuntimeWarnings (empty means, etc.); silenced once, here.

RNG_SEED = 42

import hashlib

def stable_seed(*parts):
    digest = hashlib.md5(repr(parts).encode()).hexdigest()
    return int(digest, 16) % 9999
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)

if Path.cwd().name == "scripts":
    os.chdir("..")
BASE = Path.cwd().resolve()
print("Project root:", BASE)

OUT_DIR = BASE / "outputs/codel_ipd_brain_extension"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_CSV = OUT_DIR / "master_patient_table.csv"

# Encoders
ABMIL_ENCODERS = ["uni2", "conch", "hoptimus"]   # encoders in the ABMIL predictor ensemble
OOD_ENCODERS   = ["uni2", "hoptimus"]            # encoders used for OOD signals
FOLDS = [0, 1, 2, 3, 4]
CLASS_NAMES = ("gbm", "astrocytoma", "oligodendroglioma")   # OOD density model classes
NUM_CLASSES = len(CLASS_NAMES)
ABMIL_N_CLASSES = 2                              # IDH wildtype / mutant

# OOD slide-level aggregations to precompute per encoder
OOD_AGGREGATIONS = ["topk", "attention", "mean"]

# Calibration centres (define the calibration split; no labels used for abstention)
CALIBRATION_SITES = {
    "University of Florida",
    "Milan - Italy, Fondazione IRCCS Instituto Neuroligico C. Besta",
    "Fondazione-Besta",
    "Mayo Clinic - Rochester",
}  # same fix as the primary codel notebook -- used here only to refit
   # the TCGA-locked threshold this notebook transfers to IPD Brain

TARGET_ABSTENTION_RATE = 0.20

# ABMIL checkpoints live at outputs/abmil/<model_name>/fold<k>/  -> attention source.
# e.g. {"uni2": "uni2_abmil"}).
ABMIL_DIRS = {e: e for e in ABMIL_ENCODERS}
EXTRACT_ATTENTION = True   # False -> skip attention-weighted OOD strategies cleanly

# Canonical metadata source columns 
SITE_COL   = "tissue_source_site.name"
RACE_COL   = "demographic.race"
SEX_COL    = "demographic.gender"
TUMOUR_COL = "tumour_type_who2021"
IDH_COL    = "idh_status"
print("Config loaded.")


Project root: /cs/student/project_msc/2025/aibh/mpapageo
Config loaded.


## Part A: build the IPD Brain codel table, ATRX-tiered

Same reuse pattern as the primary notebook: `load_abmil_ensemble()`
pointed at the codel embeddings tree (`ipd_brain` this time); OOD
`topk`/`mean` reused from the main bench's `ipd_brain_ood_infer.py` output
filtered to the 138 codel patients (classifier-independent, verified
identical); OOD `attention` from the codel-specific run;
MC-dropout/deep-ensemble/Laplace loaded from the already-computed
`*_codel/ipd_brain/` cache; ATRX tier merged in for the confident/full
split used below.

In [ ]:
import numpy as np
import pandas as pd


def binary_entropy(p, eps=1e-10):
    """
    Binary entropy for mutant probabilities.

    Accepts scalars or arrays and returns values with the same shape.
    Natural logarithms are used, so entropy is measured in nats.
    """
    p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1.0 - eps)

    return -(
        p * np.log(p)
        + (1.0 - p) * np.log(1.0 - p)
    )


def predictive_entropy_from_two_class_probs(probs, eps=1e-10):
    """
    Entropy of two-class probability vectors with shape (n, 2).
    """
    probs = np.asarray(probs, dtype=float)
    probs = np.clip(probs, eps, 1.0 - eps)

    return -np.sum(
        probs * np.log(probs),
        axis=1,
    )


def load_abmil_ensemble():
    """
    Load per-encoder, per-fold ABMIL predictions and derive patient-level
    uncertainty signals.

    Decomposition (all in nats):

        ensemble_entropy = aleatoric + total_mi

    where aleatoric is the mean entropy of the individual encoder-fold
    members and total_mi is their disagreement (mutual information).
    With balanced folds per encoder this further decomposes as

        total_mi = between_encoder_mi + mean_within_encoder_mi

    The same identity holds per encoder:

        entropy_<enc> = aleatoric_<enc> + within_encoder_mi_<enc>

    Epistemic uncertainty here reflects fold resampling and encoder choice
    only; architecture, hyperparameters and training recipe are fixed
    across members.
    """
    frames = []

    for enc in ABMIL_ENCODERS:
        for fold in FOLDS:
            path = (
                BASE
                / "outputs/embeddings_codel/ipd_brain"
                / enc
                / f"fold{fold}.npz"
            )

            if not path.exists():
                print("WARNING missing", path)
                continue

            d = np.load(path, allow_pickle=True)

            frames.append(
                pd.DataFrame({
                    "patient": d["patients"].astype(str),
                    "encoder": enc,
                    "fold": fold,
                    "prob_wt": d["probs"][:, 0],
                    "prob_mutant": d["probs"][:, 1],
                    "label": d["labels"].astype(int),
                })
            )

    if not frames:
        raise FileNotFoundError(
            "No ABMIL prediction (.npz) files found."
        )

    allp = pd.concat(
        frames,
        ignore_index=True,
    )

    inconsistent_labels = (
        allp.groupby("patient")["label"].nunique()
        > 1
    )

    if inconsistent_labels.any():
        bad_patients = inconsistent_labels[
            inconsistent_labels
        ].index.tolist()

        raise RuntimeError(
            "Inconsistent labels across ABMIL files for "
            f"{len(bad_patients)} patients."
        )

    # ------------------------------------------------------------
    # 1. Overall prediction and entropy across all encoder-fold models
    # ------------------------------------------------------------

    overall = (
        allp.groupby("patient")
        .agg(
            prob_wt=("prob_wt", "mean"),
            prob_mutant=("prob_mutant", "mean"),
            label=("label", "first"),
            n_abmil_rows=("prob_mutant", "size"),
            n_encoders=("encoder", "nunique"),
        )
        .reset_index()
    )

    overall["ensemble_entropy"] = (
        predictive_entropy_from_two_class_probs(
            overall[
                ["prob_wt", "prob_mutant"]
            ].to_numpy()
        )
    )

    # ------------------------------------------------------------
    # 2. Aleatoric component and total disagreement
    # ------------------------------------------------------------

    mean_member_entropy = (
        allp.assign(
            member_entropy=binary_entropy(
                allp["prob_mutant"].to_numpy()
            )
        )
        .groupby("patient")["member_entropy"]
        .mean()
        .rename("mean_member_entropy")
        .reset_index()
    )

    overall = overall.merge(
        mean_member_entropy,
        on="patient",
        how="left",
    )

    # Aleatoric uncertainty: expected entropy of the individual members.
    overall["aleatoric"] = overall["mean_member_entropy"]

    raw_total_mi = (
        overall["ensemble_entropy"]
        - overall["mean_member_entropy"]
    )

    # Flag rows where the raw value was negative before clipping, since
    # for those rows aleatoric != ensemble_entropy - total_mi.
    overall["total_mi_clipped"] = raw_total_mi < 0

    overall["total_mi"] = raw_total_mi.clip(lower=0.0)

    # ------------------------------------------------------------
    # 3. Encoder-level mean probabilities
    # ------------------------------------------------------------

    encoder_means_long = (
        allp.groupby(
            ["patient", "encoder"],
            as_index=False,
        )
        .agg(
            encoder_prob_mutant=("prob_mutant", "mean"),
            n_folds=("fold", "nunique"),
        )
    )

    encoder_means_wide = (
        encoder_means_long
        .pivot(
            index="patient",
            columns="encoder",
            values="encoder_prob_mutant",
        )
        .add_prefix("prob_mutant_")
        .reset_index()
    )

    # ------------------------------------------------------------
    # 4. Within-encoder fold disagreement
    # ------------------------------------------------------------

    within_encoder_rows = []

    for (
        patient,
        encoder,
    ), group in allp.groupby(
        ["patient", "encoder"],
        sort=False,
    ):
        fold_probs = (
            group["prob_mutant"]
            .to_numpy(dtype=float)
        )

        mean_prob = np.mean(fold_probs)

        predictive_entropy = binary_entropy(
            mean_prob
        )

        expected_entropy = np.mean(
            binary_entropy(fold_probs)
        )

        within_mi = max(
            float(
                predictive_entropy
                - expected_entropy
            ),
            0.0,
        )

        within_encoder_rows.append({
            "patient": patient,
            "encoder": encoder,
            "within_encoder_entropy":
                float(predictive_entropy),
            "within_encoder_aleatoric":
                float(expected_entropy),
            "within_encoder_mi": within_mi,
            "fold_probability_sd":
                float(np.std(fold_probs, ddof=0)),
            "fold_probability_range":
                float(
                    np.max(fold_probs)
                    - np.min(fold_probs)
                ),
            "n_folds": len(fold_probs),
        })

    within_encoder = pd.DataFrame(
        within_encoder_rows
    )

    def _pivot(column, prefix):
        return (
            within_encoder
            .pivot(
                index="patient",
                columns="encoder",
                values=column,
            )
            .add_prefix(prefix)
            .reset_index()
        )

    within_mi_wide = _pivot(
        "within_encoder_mi",
        "within_encoder_mi_",
    )

    within_sd_wide = _pivot(
        "fold_probability_sd",
        "fold_probability_sd_",
    )

    within_entropy_wide = _pivot(
        "within_encoder_entropy",
        "entropy_",
    )

    within_aleatoric_wide = _pivot(
        "within_encoder_aleatoric",
        "aleatoric_",
    )

    mean_within_encoder = (
        within_encoder
        .groupby("patient")
        .agg(
            mean_within_encoder_mi=(
                "within_encoder_mi",
                "mean",
            ),
            max_within_encoder_mi=(
                "within_encoder_mi",
                "max",
            ),
            mean_fold_probability_sd=(
                "fold_probability_sd",
                "mean",
            ),
            max_fold_probability_sd=(
                "fold_probability_sd",
                "max",
            ),
        )
        .reset_index()
    )

    # ------------------------------------------------------------
    # 5. Between-encoder disagreement
    # ------------------------------------------------------------

    between_encoder_rows = []

    for patient, group in encoder_means_long.groupby(
        "patient",
        sort=False,
    ):
        encoder_probs = (
            group["encoder_prob_mutant"]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(encoder_probs) == 0:
            between_encoder_rows.append({
                "patient": patient,
                "between_encoder_entropy": np.nan,
                "between_encoder_mi": np.nan,
                "encoder_probability_sd": np.nan,
                "encoder_probability_range": np.nan,
                "n_available_encoders": 0,
            })
            continue

        mean_prob = np.mean(encoder_probs)

        between_entropy = binary_entropy(
            mean_prob
        )

        mean_encoder_entropy = np.mean(
            binary_entropy(encoder_probs)
        )

        between_mi = max(
            float(
                between_entropy
                - mean_encoder_entropy
            ),
            0.0,
        )

        between_encoder_rows.append({
            "patient": patient,
            "between_encoder_entropy":
                float(between_entropy),
            "between_encoder_mi":
                between_mi,
            "encoder_probability_sd":
                float(
                    np.std(
                        encoder_probs,
                        ddof=0,
                    )
                ),
            "encoder_probability_range":
                float(
                    np.max(encoder_probs)
                    - np.min(encoder_probs)
                ),
            "n_available_encoders":
                len(encoder_probs),
        })

    between_encoder = pd.DataFrame(
        between_encoder_rows
    )

    # ------------------------------------------------------------
    # 6. Merge patient-level outputs
    # ------------------------------------------------------------

    output = (
        overall
        .merge(
            encoder_means_wide,
            on="patient",
            how="left",
        )
        .merge(
            within_mi_wide,
            on="patient",
            how="left",
        )
        .merge(
            within_sd_wide,
            on="patient",
            how="left",
        )
        .merge(
            within_entropy_wide,
            on="patient",
            how="left",
        )
        .merge(
            within_aleatoric_wide,
            on="patient",
            how="left",
        )
        .merge(
            mean_within_encoder,
            on="patient",
            how="left",
        )
        .merge(
            between_encoder,
            on="patient",
            how="left",
        )
    )

    # ------------------------------------------------------------
    # 7. Consistency checks on the decomposition
    # ------------------------------------------------------------

    alea_dev = (
        (output["ensemble_entropy"] - output["total_mi"])
        - output["aleatoric"]
    ).abs()

    decomp_dev = (
        output["total_mi"]
        - (
            output["between_encoder_mi"]
            + output["mean_within_encoder_mi"]
        )
    ).abs()

    print(
        "decomposition checks | "
        f"aleatoric max dev {alea_dev.max():.2e} "
        f"(clipped rows {int(output['total_mi_clipped'].sum())}) | "
        f"total vs between+within max dev {decomp_dev.max():.2e}"
    )
    print(
        "members per patient  "
        f"{output['n_abmil_rows'].value_counts().to_dict()} | "
        "encoders per patient "
        f"{output['n_encoders'].value_counts().to_dict()}"
    )

    return output, allp, within_encoder

In [3]:
abmil, abmil_member_predictions, within_encoder_results = load_abmil_ensemble()

ood_frames = []
for enc in OOD_ENCODERS:
    main_ipd = pd.read_csv(BASE / f"outputs/ood/ipd_brain/{enc}_summary.csv")
    main_ipd["patient"] = main_ipd["patient"].astype(str)
    topk_mean = main_ipd[["patient", f"ood_topk_{enc}", f"ood_mean_{enc}"]]
    attn = pd.read_csv(BASE / f"outputs/ood/codel_ipd_brain/{enc}_summary.csv")
    attn["patient"] = attn["patient"].astype(str)
    attn = attn[["patient", f"ood_attention_{enc}"]]
    ood_frames.append(topk_mean.merge(attn, on="patient", how="outer"))
ood = ood_frames[0]
for f in ood_frames[1:]:
    ood = ood.merge(f, on="patient", how="outer")

labels = pd.read_csv(BASE / "data/raw/ipd_brain/clinical/ipd_brain_1p19q_labels.csv")
labels["case_id"] = labels["case_id"].astype(str)
labels["atrx_norm"] = labels["ATRX"].astype(str).str.strip().str.lower()
labels["atrx_concordant"] = (
    ((labels["codel_binary"] == 1) & (labels["atrx_norm"] == "retained"))
    | ((labels["codel_binary"] == 0) & labels["atrx_norm"].str.contains("not retained|loss|lost", na=False))
)
print("ATRX concordance:", labels["atrx_concordant"].sum(), "/", len(labels))

master = (abmil.merge(ood, on="patient", how="inner")
               .merge(labels.rename(columns={"case_id": "patient"})[["patient", "atrx_concordant", "ATRX"]],
                      on="patient", how="left"))
master.to_csv(MASTER_CSV, index=False)
print("Saved", MASTER_CSV, master.shape)


decomposition checks | aleatoric max dev 0.00e+00 (clipped rows 0) | total vs between+within max dev 1.35e-07
members per patient  {15: 138} | encoders per patient {3: 138}
ATRX concordance: 127 / 138
Saved /cs/student/project_msc/2025/aibh/mpapageo/outputs/codel_ipd_brain_extension/master_patient_table.csv (138, 43)


### A.6/A.7/A.8: MC-dropout / deep-ensemble / Laplace (verbatim, path-adapted to ipd_brain)

In [ ]:
MC_DROPOUT_COLS_EXPECTED = (
    [f"mc_predictive_entropy_{e}" for e in ABMIL_ENCODERS]
    + [f"mc_aleatoric_{e}" for e in ABMIL_ENCODERS]
    + [f"mc_epistemic_mi_{e}" for e in ABMIL_ENCODERS]
    + ["mc_predictive_entropy", "mc_aleatoric", "mc_epistemic_mi"]
)

if all(c in master.columns for c in MC_DROPOUT_COLS_EXPECTED):
    print("MC-dropout columns already present in the cached master table; skipping recomputation.")
else:
    print("MC-dropout columns missing; loading from outputs/mc_dropout_codel/ipd_brain/...")

    mc_frames = []
    for enc in ABMIL_ENCODERS:
        for fold in FOLDS:
            p = BASE / f"outputs/mc_dropout_codel/ipd_brain/{enc}/fold{fold}_summary.csv"
            if not p.exists():
                print(f"  WARNING missing {p}"); continue
            fdf = pd.read_csv(p)
            fdf["encoder"] = enc
            mc_frames.append(fdf)
    mc_all = pd.concat(mc_frames, ignore_index=True)
    mc_all["patient"] = mc_all["patient"].astype(str)

    per_enc_prob = (mc_all.groupby(["patient", "encoder"])["mean_prob_mutant"]
                     .mean().reset_index())
    per_enc_prob["mc_predictive_entropy"] = binary_entropy(
        per_enc_prob["mean_prob_mutant"].to_numpy()
    )
    per_enc_linear = (mc_all.groupby(["patient", "encoder"])
                       .agg(mc_aleatoric=("aleatoric", "mean"),
                            mc_epistemic_mi=("epistemic_mi", "mean"))
                       .reset_index())
    per_enc = per_enc_prob[["patient", "encoder", "mc_predictive_entropy"]].merge(
        per_enc_linear, on=["patient", "encoder"]
    )

    wide = per_enc.pivot(index="patient", columns="encoder",
                          values=["mc_predictive_entropy", "mc_aleatoric", "mc_epistemic_mi"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    # Pooled mc_aleatoric / mc_epistemic_mi: naive mean-of-3-encoder-means is exact
    # here (each encoder already averages an equal-sized 5-fold group, so averaging
    # 3 equal-sized group means reproduces the true 15-fold mean for these *linear*
    # aggregates). mc_predictive_entropy is NOT linear in this way -- recomputed
    # from the true grand-pooled probability across all 15 (3 encoder x 5 fold)
    # fold-means, same principle as the per-encoder fix above, one level up.
    linear_pooled = (per_enc.groupby("patient")
                      .agg(mc_aleatoric=("mc_aleatoric", "mean"),
                           mc_epistemic_mi=("mc_epistemic_mi", "mean"))
                      .reset_index())

    grand_mean_prob = mc_all.groupby("patient")["mean_prob_mutant"].mean()
    grand_pooled_entropy = pd.DataFrame({
        "patient": grand_mean_prob.index,
        "mc_predictive_entropy": binary_entropy(grand_mean_prob.to_numpy()),
    })

    pooled = linear_pooled.merge(grand_pooled_entropy, on="patient", how="left")

    mc_cols = wide.merge(pooled, on="patient", how="outer")

    master["patient"] = master["patient"].astype(str)
    # Drop any stale mc_* columns from a previous partial run before merging --
    # otherwise a name collision silently produces _x/_y duplicate columns and
    # the clean per-encoder names (e.g. mc_predictive_entropy_uni2) disappear,
    # which breaks every predictor except "ensemble" without raising an error.
    master = master.drop(columns=[c for c in master.columns if c.startswith("mc_")],
                          errors="ignore")
    master = master.merge(mc_cols, on="patient", how="left")

    backup = MASTER_CSV.with_suffix(".pre_mc_dropout.csv")
    if not backup.exists():
        pd.read_csv(MASTER_CSV).to_csv(backup, index=False)
    master.to_csv(MASTER_CSV, index=False)
    print("appended MC-dropout columns:", [c for c in master.columns if c.startswith("mc_")])


MC-dropout columns missing; loading from outputs/mc_dropout_codel/ipd_brain/...
appended MC-dropout columns: ['mc_predictive_entropy_conch', 'mc_predictive_entropy_hoptimus', 'mc_predictive_entropy_uni2', 'mc_aleatoric_conch', 'mc_aleatoric_hoptimus', 'mc_aleatoric_uni2', 'mc_epistemic_mi_conch', 'mc_epistemic_mi_hoptimus', 'mc_epistemic_mi_uni2', 'mc_aleatoric', 'mc_epistemic_mi', 'mc_predictive_entropy']


In [ ]:
DEEP_ENSEMBLE_COLS_EXPECTED = (
    [f"de_predictive_entropy_{e}" for e in ABMIL_ENCODERS]
    + [f"de_aleatoric_{e}" for e in ABMIL_ENCODERS]
    + [f"de_within_fold_mi_{e}" for e in ABMIL_ENCODERS]
    + ["de_predictive_entropy", "de_aleatoric", "de_within_fold_mi"]
)

if all(c in master.columns for c in DEEP_ENSEMBLE_COLS_EXPECTED):
    print("Deep-ensemble columns already present in the cached master table; skipping recomputation.")
else:
    print("Deep-ensemble columns missing; loading from outputs/deep_ensemble_codel/ipd_brain/...")

    def _de_binary_entropy(p, eps=1e-10):
        p = np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)
        return -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))

    de_per_enc_frames = []
    de_consistency = []
    all_fold_rows = []
    for enc in ABMIL_ENCODERS:
        fold_rows = []
        for fold in FOLDS:
            p = BASE / f"outputs/deep_ensemble_codel/ipd_brain/{enc}/fold{fold}.npz"
            if not p.exists():
                print(f"  WARNING missing {p}"); continue
            d = np.load(p, allow_pickle=True)
            patients = d["patients"].astype(str)
            member_probs = d["member_probs"]              # [N, K] init-seed members
            fold_mean = member_probs.mean(axis=1)
            fold_pred_entropy = _de_binary_entropy(fold_mean)
            fold_aleatoric = _de_binary_entropy(member_probs).mean(axis=1)
            fold_mi = np.clip(fold_pred_entropy - fold_aleatoric, 0.0, None)
            df = pd.DataFrame({
                "patient": patients, "fold": fold,
                "fold_mean_prob": fold_mean,
                "fold_pred_entropy": fold_pred_entropy,
                "fold_aleatoric": fold_aleatoric,
                "fold_within_fold_mi": fold_mi,
            })
            fold_rows.append(df)
            all_fold_rows.append(df.assign(encoder=enc))
        allf = pd.concat(fold_rows, ignore_index=True)

        # de_predictive_entropy_<enc>: same Jensen's-inequality issue fixed for
        # mc_dropout/laplace applies here too -- recompute from this encoder's
        # own grand-mean probability across its 5 folds, not the mean of 5
        # already-computed fold-level entropies.
        enc_grand_mean_prob = allf.groupby("patient")["fold_mean_prob"].mean()
        per_patient = allf.groupby("patient").agg(
            de_aleatoric=("fold_aleatoric", "mean"),
            de_within_fold_mi=("fold_within_fold_mi", "mean"),
        ).reset_index()
        per_patient["de_predictive_entropy"] = per_patient["patient"].map(
            pd.Series(_de_binary_entropy(enc_grand_mean_prob.to_numpy()),
                      index=enc_grand_mean_prob.index)
        )
        per_patient["encoder"] = enc

        grand = allf.groupby("patient").agg(
            grand_mean_prob=("fold_mean_prob", "mean"),
            mean_fold_entropy=("fold_pred_entropy", "mean"),
        )
        between_fold_mi = np.clip(
            _de_binary_entropy(grand["grand_mean_prob"]) - grand["mean_fold_entropy"], 0.0, None
        )
        total_mi_check = (
            between_fold_mi.to_numpy()
            + per_patient.set_index("patient")["de_within_fold_mi"].reindex(grand.index).to_numpy()
        )
        de_consistency.append((enc, float(total_mi_check.max()), float(total_mi_check.min())))

        de_per_enc_frames.append(per_patient)

    de_all = pd.concat(de_per_enc_frames, ignore_index=True)
    print("deep-ensemble between+within MI range per encoder (sanity only, not used downstream):",
          de_consistency)

    wide = de_all.pivot(index="patient", columns="encoder",
                         values=["de_predictive_entropy", "de_aleatoric", "de_within_fold_mi"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    linear_pooled = (de_all.groupby("patient")
                      .agg(de_aleatoric=("de_aleatoric", "mean"),
                           de_within_fold_mi=("de_within_fold_mi", "mean"))
                      .reset_index())

    all_fold_df = pd.concat(all_fold_rows, ignore_index=True)
    grand_mean_prob = all_fold_df.groupby("patient")["fold_mean_prob"].mean()
    grand_pooled_entropy = pd.DataFrame({
        "patient": grand_mean_prob.index,
        "de_predictive_entropy": _de_binary_entropy(grand_mean_prob.to_numpy()),
    })

    pooled = linear_pooled.merge(grand_pooled_entropy, on="patient", how="left")

    de_cols = wide.merge(pooled, on="patient", how="outer")

    master["patient"] = master["patient"].astype(str)
    master = master.drop(columns=[c for c in master.columns if c.startswith("de_")],
                          errors="ignore")
    master = master.merge(de_cols, on="patient", how="left")

    backup = MASTER_CSV.with_suffix(".pre_deep_ensemble.csv")
    if not backup.exists():
        pd.read_csv(MASTER_CSV).to_csv(backup, index=False)
    master.to_csv(MASTER_CSV, index=False)
    print("appended deep-ensemble columns:", [c for c in master.columns if c.startswith("de_")])


Deep-ensemble columns missing; loading from outputs/deep_ensemble_codel/ipd_brain/...
deep-ensemble between+within MI range per encoder (sanity only, not used downstream): [('uni2', 0.15179270057525238, 0.0016637629238884124), ('conch', 0.10156190798227234, 0.0235732005402594), ('hoptimus', 0.2813430843722627, 0.007821265491306831)]
appended deep-ensemble columns: ['de_predictive_entropy_conch', 'de_predictive_entropy_hoptimus', 'de_predictive_entropy_uni2', 'de_aleatoric_conch', 'de_aleatoric_hoptimus', 'de_aleatoric_uni2', 'de_within_fold_mi_conch', 'de_within_fold_mi_hoptimus', 'de_within_fold_mi_uni2', 'de_aleatoric', 'de_within_fold_mi', 'de_predictive_entropy']


In [ ]:
LAPLACE_COLS_EXPECTED = (
    [f"laplace_predictive_entropy_{e}" for e in ABMIL_ENCODERS]
    + [f"laplace_aleatoric_{e}" for e in ABMIL_ENCODERS]
    + [f"laplace_epistemic_mi_{e}" for e in ABMIL_ENCODERS]
    + ["laplace_predictive_entropy", "laplace_aleatoric", "laplace_epistemic_mi"]
)

if all(c in master.columns for c in LAPLACE_COLS_EXPECTED):
    print("Laplace columns already present in the cached master table; skipping recomputation.")
else:
    print("Laplace columns missing; loading from outputs/laplace_codel/ipd_brain/...")

    laplace_frames = []
    for enc in ABMIL_ENCODERS:
        for fold in FOLDS:
            p = BASE / f"outputs/laplace_codel/ipd_brain/{enc}/fold{fold}_summary.csv"
            if not p.exists():
                print(f"  WARNING missing {p}"); continue
            fdf = pd.read_csv(p)
            fdf["encoder"] = enc
            laplace_frames.append(fdf)
    laplace_all = pd.concat(laplace_frames, ignore_index=True)
    laplace_all["patient"] = laplace_all["patient"].astype(str)

    per_enc_prob = (laplace_all.groupby(["patient", "encoder"])["mean_prob_mutant"]
                     .mean().reset_index())
    per_enc_prob["laplace_predictive_entropy"] = binary_entropy(
        per_enc_prob["mean_prob_mutant"].to_numpy()
    )
    per_enc_linear = (laplace_all.groupby(["patient", "encoder"])
                       .agg(laplace_aleatoric=("aleatoric", "mean"),
                            laplace_epistemic_mi=("epistemic_mi", "mean"))
                       .reset_index())
    per_enc = per_enc_prob[["patient", "encoder", "laplace_predictive_entropy"]].merge(
        per_enc_linear, on=["patient", "encoder"]
    )

    wide = per_enc.pivot(index="patient", columns="encoder",
                          values=["laplace_predictive_entropy", "laplace_aleatoric", "laplace_epistemic_mi"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    # Pooled laplace_aleatoric / laplace_epistemic_mi: naive mean-of-3-encoder-means
    # is exact here (linear aggregate over equal-sized groups). laplace_predictive_entropy
    # is recomputed from the true grand-pooled probability across all 15
    # (3 encoder x 5 fold) fold-means, same principle as the per-encoder fix above.
    linear_pooled = (per_enc.groupby("patient")
                      .agg(laplace_aleatoric=("laplace_aleatoric", "mean"),
                           laplace_epistemic_mi=("laplace_epistemic_mi", "mean"))
                      .reset_index())

    grand_mean_prob = laplace_all.groupby("patient")["mean_prob_mutant"].mean()
    grand_pooled_entropy = pd.DataFrame({
        "patient": grand_mean_prob.index,
        "laplace_predictive_entropy": binary_entropy(grand_mean_prob.to_numpy()),
    })

    pooled = linear_pooled.merge(grand_pooled_entropy, on="patient", how="left")

    laplace_cols = wide.merge(pooled, on="patient", how="outer")

    master["patient"] = master["patient"].astype(str)
    master = master.drop(columns=[c for c in master.columns if c.startswith("laplace_")],
                          errors="ignore")
    master = master.merge(laplace_cols, on="patient", how="left")

    backup = MASTER_CSV.with_suffix(".pre_laplace.csv")
    if not backup.exists():
        pd.read_csv(MASTER_CSV).to_csv(backup, index=False)
    master.to_csv(MASTER_CSV, index=False)
    print("appended Laplace columns:", [c for c in master.columns if c.startswith("laplace_")])


Laplace columns missing; loading from outputs/laplace_codel/ipd_brain/...
appended Laplace columns: ['laplace_predictive_entropy_conch', 'laplace_predictive_entropy_hoptimus', 'laplace_predictive_entropy_uni2', 'laplace_aleatoric_conch', 'laplace_aleatoric_hoptimus', 'laplace_aleatoric_uni2', 'laplace_epistemic_mi_conch', 'laplace_epistemic_mi_hoptimus', 'laplace_epistemic_mi_uni2', 'laplace_aleatoric', 'laplace_epistemic_mi', 'laplace_predictive_entropy']


## Evaluation: TCGA-transferred locked threshold, both tiers

Threshold locked on the primary codel notebook's own TCGA calibration
sites, transferred unchanged to IPD Brain, mirroring the main bench's IPD
extension's own TCGA-transferred, no-refitting primary combo. Reports the
family-level AURC comparison for the `ensemble` predictor on both the
ATRX-concordant tier (n=127, headline) and the full 138 (robustness
check).

In [7]:
# import kendalltau
from scipy.stats import kendalltau
N_BOOT_AURC = 2000   # renamed from N_BOOT to avoid colliding with C5's own N_BOOT above
COVERAGES = (0.9, 0.8, 0.7, 0.5)
TARGET_COV = 0.80
OOD_AGGS = ("topk", "attention", "mean", "maha_topk", "maha_attention", "maha_mean")

# --- per-predictor signal sets ----------------------------------------------
def signal_families(predictor):
    """Signals matched to each predictor's own members."""
    if predictor == "ensemble":
        return {
            "predictive": ["ensemble_entropy", "aleatoric"],
            "epistemic":  ["total_mi", "mean_within_encoder_mi", "between_encoder_mi"],
            "ood":        [f"ood_{a}_{e}" for e in ("uni2", "conch", "hoptimus")
                           for a in OOD_AGGS],
            "mc_dropout":    ["mc_predictive_entropy", "mc_aleatoric", "mc_epistemic_mi"],
            "deep_ensemble": ["de_predictive_entropy", "de_aleatoric", "de_within_fold_mi"],
            "laplace":       ["laplace_predictive_entropy", "laplace_aleatoric", "laplace_epistemic_mi"],
        }
    encs = predictor.split("+")
    return {
        "predictive": [f"entropy_{e}" for e in encs] + [f"aleatoric_{e}" for e in encs],
        "epistemic":  [f"within_encoder_mi_{e}" for e in encs],
        "ood":        [f"ood_{a}_{e}" for e in encs for a in OOD_AGGS],
        "mc_dropout":    ([f"mc_predictive_entropy_{e}" for e in encs]
                          + [f"mc_aleatoric_{e}" for e in encs]
                          + [f"mc_epistemic_mi_{e}" for e in encs]),
        "deep_ensemble": ([f"de_predictive_entropy_{e}" for e in encs]
                          + [f"de_aleatoric_{e}" for e in encs]
                          + [f"de_within_fold_mi_{e}" for e in encs]),
        "laplace":       ([f"laplace_predictive_entropy_{e}" for e in encs]
                          + [f"laplace_aleatoric_{e}" for e in encs]
                          + [f"laplace_epistemic_mi_{e}" for e in encs]),
    }

def resolve(families, df):
    present = {f: [c for c in cols if c in df.columns] for f, cols in families.items()}
    missing = {f: [c for c in cols if c not in df.columns] for f, cols in families.items()}
    return {f: c for f, c in present.items() if c}, {f: c for f, c in missing.items() if c}

def common_cohort(df, signals, extra):
    cols = [c for c in signals if c in df.columns] + [c for c in extra if c in df.columns]
    keep = df[cols].replace([np.inf, -np.inf], np.nan).dropna().index
    return df.loc[keep]

# --- risk-coverage machinery ------------------------------------------------
def risk_coverage(score, errors):
    order = np.argsort(np.asarray(score, float), kind="stable")
    e = np.asarray(errors, int)[order]
    k = np.arange(1, len(e) + 1)
    return k / len(e), np.cumsum(e) / k

def aurc(score, errors):
    return float(np.mean(risk_coverage(score, errors)[1]))

def eaurc(score, errors):
    errors = np.asarray(errors, int)
    return aurc(score, errors) - aurc(errors.astype(float), errors)

def risk_at_coverage(score, errors, cov):
    sr = risk_coverage(score, errors)[1]
    return float(sr[max(0, int(np.ceil(cov * len(sr))) - 1)])

def coverage_at_risk(score, errors, target):
    cov, sr = risk_coverage(score, errors)
    ok = np.flatnonzero(sr <= target)
    return float(cov[ok[-1]]) if len(ok) else 0.0

# --- bootstrap machinery: shared resamples so AURC diffs can be paired -------
def boot_indices(n, n_boot=N_BOOT_AURC, seed=0):
    rng = np.random.default_rng(seed)
    return rng.integers(0, n, size=(n_boot, n))

def aurc_boot(score, errors, idx):
    score, errors = np.asarray(score, float), np.asarray(errors, int)
    return np.array([aurc(score[i], errors[i]) for i in idx])

# --- signal table -----------------------------------------------------------
def evaluate_signals(df, prob_col, thr, families, idx):
    y = df["label"].astype(int).to_numpy()
    err = ((df[prob_col].astype(float).to_numpy() >= thr).astype(int) != y).astype(int)
    base = err.mean()
    rows, draws = [], {}
    for fam, cols in families.items():
        for c in cols:
            s = df[c].astype(float).to_numpy()
            b = aurc_boot(s, err, idx)
            draws[c] = b
            rows.append({"family": fam, "signal": c, "aurc": aurc(s, err),
                         "aurc_lo": float(np.percentile(b, 2.5)),
                         "aurc_hi": float(np.percentile(b, 97.5)),
                         "eaurc": eaurc(s, err),
                         "frac_gap_closed": 1 - eaurc(s, err) / (base - aurc(err.astype(float), err)),
                         "cov_at_half_risk": coverage_at_risk(s, err, base / 2),
                         **{f"risk@{c2:.0%}": risk_at_coverage(s, err, c2) for c2 in COVERAGES}})
    rows.append({"family": "reference", "signal": "random", "aurc": base,
                 "aurc_lo": np.nan, "aurc_hi": np.nan,
                 "eaurc": base - aurc(err.astype(float), err), "frac_gap_closed": 0.0,
                 "cov_at_half_risk": np.nan,
                 **{f"risk@{c2:.0%}": base for c2 in COVERAGES}})
    return pd.DataFrame(rows).sort_values(["family", "aurc"]), draws, err

def paired_contrast(draws, a, b):
    """Paired bootstrap CI on AURC(a) - AURC(b). Negative favours a."""
    d = draws[a] - draws[b]
    return {"contrast": f"{a} - {b}", "mean_diff": float(d.mean()),
            "lo": float(np.percentile(d, 2.5)), "hi": float(np.percentile(d, 97.5)),
            "p_favours_a": float((d < 0).mean())}

# --- what drives each signal: error, or acquisition? ------------------------
def signal_structure(df, err, signals, group_col="site"):
    rows = []
    for c in signals:
        s = df[c].astype(float)
        grand = s.mean()
        ss_between = sum(len(g) * (g[c].mean() - grand) ** 2 for _, g in df.groupby(group_col))
        rows.append({"signal": c,
                     "tau_with_error": kendalltau(s, err, nan_policy="omit").statistic,
                     f"var_by_{group_col}": ss_between / ((s - grand) ** 2).sum()})
    return pd.DataFrame(rows).sort_values("tau_with_error", ascending=False)

# --- mixing sweep (oracle bound: minimum located on eval) -------------------
def mixing_sweep(df, err, pred_col, ood_col, weights=np.linspace(0, 1, 11)):
    z = lambda c: ((df[c] - df[c].mean()) / df[c].std(ddof=0)).to_numpy()
    zp, zo = z(pred_col), z(ood_col)
    return pd.DataFrame([{"w_ood": w, "aurc": aurc((1 - w) * zp + w * zo, err)}
                         for w in weights])

# --- calibration-locked operating point -------------------------------------
def lock_and_apply(cal, ev, col, target_cov, prob_col, thr):
    mu, sd = float(cal[col].mean()), float(cal[col].std(ddof=0))
    cutoff = float(np.quantile((cal[col] - mu) / sd, target_cov))
    retain = ((ev[col] - mu) / sd <= cutoff).to_numpy()
    y = ev["label"].astype(int).to_numpy()
    err = ((ev[prob_col].astype(float).to_numpy() >= thr).astype(int) != y).astype(int)
    return {"signal": col, "cutoff": cutoff, "achieved_coverage": float(retain.mean()),
            "coverage_shift": float(retain.mean() - target_cov),
            "risk_full": float(err.mean()),
            "risk_retained": float(err[retain].mean()) if retain.any() else np.nan,
            "risk_deferred": float(err[~retain].mean()) if (~retain).any() else np.nan}



codel_tcga_master = pd.read_csv(BASE / "outputs/selective_prediction_bench_codel/master_patient_table.csv")
codel_tcga_site = (codel_tcga_master["tissue_source_site.name"].fillna("Unknown").astype(str).str.strip())
codel_tcga_cal = codel_tcga_master[codel_tcga_site.isin(CALIBRATION_SITES)]
fpr, tpr, thr_grid = roc_curve(codel_tcga_cal["label"], codel_tcga_cal["prob_mutant"])
LOCKED_THR = float(thr_grid[np.argmax(tpr - fpr)])
print(f"TCGA-transferred locked threshold (ensemble): {LOCKED_THR:.4f} (fit on n={len(codel_tcga_cal)} codel calibration patients)")

fams_ipd, missing_ipd = resolve(signal_families("ensemble"), master)
sig_cols_ipd = [c for cols in fams_ipd.values() for c in cols]
if missing_ipd:
    print("MISSING (excluded):", [c for cols in missing_ipd.values() for c in cols])

for tier_name, tier_mask in [("ATRX-concordant (confident tier)", master["atrx_concordant"] == True),
                              ("Full cohort (incl. 11 ATRX-discordant)", pd.Series(True, index=master.index))]:
    sub = master[tier_mask]
    EV = common_cohort(sub, sig_cols_ipd, ["label", "prob_mutant"])
    print(f"\n{'='*70}\n{tier_name}: n={len(EV)}\n{'='*70}")
    idx = boot_indices(len(EV))
    tab, draws, err = evaluate_signals(EV, "prob_mutant", LOCKED_THR, fams_ipd, idx)
    display(tab.round(4))
    ref = tab.query("family==\'predictive\'").iloc[0]["signal"]
    contrasts = [paired_contrast(draws, tab.query("family==@f").iloc[0]["signal"], ref)
                 for f in fams_ipd if f != "predictive"]
    display(pd.DataFrame(contrasts).round(4))
    tab.assign(tier=tier_name).to_csv(
        OUT_DIR / f"family_aurc_{tier_name.split()[0].lower()}.csv", index=False)


TCGA-transferred locked threshold (ensemble): 0.1779 (fit on n=66 codel calibration patients)
MISSING (excluded): ['ood_maha_topk_uni2', 'ood_maha_attention_uni2', 'ood_maha_mean_uni2', 'ood_topk_conch', 'ood_attention_conch', 'ood_mean_conch', 'ood_maha_topk_conch', 'ood_maha_attention_conch', 'ood_maha_mean_conch', 'ood_maha_topk_hoptimus', 'ood_maha_attention_hoptimus', 'ood_maha_mean_hoptimus']

ATRX-concordant (confident tier): n=127


,family,signal,aurc,aurc_lo,aurc_hi,eaurc,frac_gap_closed,cov_at_half_risk,risk@90%,risk@80%,risk@70%,risk@50%
15,deep_ensemble,de_aleatoric,0.0944,0.0526,0.1476,0.0373,0.8553,0.7323,0.2609,0.2157,0.1124,0.0312
14,deep_ensemble,de_predictive_entropy,0.1011,0.0585,0.1556,0.0441,0.8291,0.7087,0.2783,0.2353,0.1573,0.0156
16,deep_ensemble,de_within_fold_mi,0.1605,0.0984,0.2390,0.1035,0.5988,0.5118,0.2957,0.2647,0.2247,0.1562
3,epistemic,mean_within_encoder_mi,0.1614,0.0903,0.2501,0.1043,0.5955,0.5827,0.2870,0.2549,0.2135,0.0938
2,epistemic,total_mi,0.1863,0.1159,0.2669,0.1293,0.4988,0.3622,0.3043,0.2843,0.2360,0.2500
4,epistemic,between_encoder_mi,0.3519,0.2290,0.4646,0.2949,-0.1433,0.0000,0.3130,0.2941,0.2921,0.2969
18,laplace,laplace_aleatoric,0.0970,0.0546,0.1518,0.0399,0.8452,0.7323,0.2696,0.2157,0.1236,0.0312
17,laplace,laplace_predictive_entropy,0.1038,0.0605,0.1589,0.0467,0.8189,0.7008,0.2783,0.2451,0.1573,0.0312
19,laplace,laplace_epistemic_mi,0.1133,0.0639,0.1773,0.0562,0.7820,0.6772,0.2696,0.2255,0.1685,0.0625
12,mc_dropout,mc_aleatoric,0.0933,0.0525,0.1457,0.0362,0.8595,0.7244,0.2609,0.2059,0.1461,0.0312


,contrast,mean_diff,lo,hi,p_favours_a
0,mean_within_encoder_mi - aleatoric,0.0695,0.0212,0.1297,0.0005
1,ood_attention_uni2 - aleatoric,0.1610,0.1031,0.2270,0.0000
2,mc_aleatoric - aleatoric,-0.0002,-0.0030,0.0020,0.5515
3,de_aleatoric - aleatoric,0.0008,-0.0040,0.0060,0.3725
4,laplace_aleatoric - aleatoric,0.0035,-0.0003,0.0081,0.0365



Full cohort (incl. 11 ATRX-discordant): n=138


,family,signal,aurc,aurc_lo,aurc_hi,eaurc,frac_gap_closed,cov_at_half_risk,risk@90%,risk@80%,risk@70%,risk@50%
15,deep_ensemble,de_aleatoric,0.1113,0.0681,0.1694,0.0410,0.8523,0.7174,0.2960,0.2523,0.1546,0.0290
14,deep_ensemble,de_predictive_entropy,0.1174,0.0720,0.1761,0.0471,0.8304,0.6812,0.3120,0.2703,0.1856,0.0435
16,deep_ensemble,de_within_fold_mi,0.1816,0.1200,0.2600,0.1113,0.5989,0.4420,0.3280,0.2973,0.2577,0.1884
3,epistemic,mean_within_encoder_mi,0.1746,0.1044,0.2636,0.1043,0.6241,0.5870,0.3200,0.2883,0.2474,0.1014
2,epistemic,total_mi,0.1937,0.1291,0.2736,0.1234,0.5552,0.3768,0.3280,0.2973,0.2680,0.2464
4,epistemic,between_encoder_mi,0.3540,0.2435,0.4710,0.2837,-0.0223,0.0000,0.3360,0.3243,0.3093,0.3188
18,laplace,laplace_aleatoric,0.1150,0.0710,0.1740,0.0447,0.8390,0.7101,0.3040,0.2613,0.1649,0.0435
17,laplace,laplace_predictive_entropy,0.1202,0.0756,0.1792,0.0499,0.8203,0.6739,0.3120,0.2793,0.1959,0.0435
19,laplace,laplace_epistemic_mi,0.1363,0.0850,0.2059,0.0660,0.7622,0.6667,0.3040,0.2613,0.2165,0.1014
12,mc_dropout,mc_aleatoric,0.1107,0.0685,0.1682,0.0403,0.8546,0.6884,0.3040,0.2432,0.1856,0.0435


,contrast,mean_diff,lo,hi,p_favours_a
0,mean_within_encoder_mi - aleatoric,0.0650,0.0176,0.1253,0.0000
1,ood_attention_uni2 - aleatoric,0.1772,0.1179,0.2449,0.0000
2,mc_aleatoric - aleatoric,-0.0003,-0.0028,0.0020,0.5665
3,de_aleatoric - aleatoric,0.0004,-0.0049,0.0056,0.4340
4,laplace_aleatoric - aleatoric,0.0041,0.0004,0.0088,0.0120


## Calibration-locked deferral, full metric panel, BH-FDR corrected

Ports the calibration-locked bootstrap/FDR machinery verbatim from the
main TCGA bench's own section (`selective_prediction_bench_clean_extended.ipynb`:
`SEL_SUITE`, `suite_on`/`defined_sel`/`suite_masked`, `perm_p_two_sided`,
`random_deferral`, `fit_calibration_cutoffs`, `selective_table_locked`),
unchanged except for the source cohort.

Cutoffs are fit on the n=66 codel TCGA calibration set (`codel_tcga_cal`,
already loaded above for `LOCKED_THR`) and transferred unchanged to IPD
Brain, the same TCGA-locked, no-refitting combo used throughout the main
bench (IPD Brain has no calibration subset of its own here; the whole
127/138-patient tier is evaluation only). Full 13-metric panel (AUROC,
AUPRC, BSS, balanced accuracy, sensitivity, specificity, PPV, NPV, F1,
Brier, ECE, cal_gap, accuracy), against both the full-cohort
(no-abstention) baseline and a random-deferral null matched to achieved
coverage, BH-FDR corrected within each metric across every family ×
coverage × tier combination, same convention as every other
calibration-locked table in this project.

In [ ]:
from scipy.stats import rankdata

SEL_SUITE = [
    "auroc", "auprc", "bss", "balanced_accuracy", "sensitivity",
    "specificity", "ppv", "npv", "f1", "brier", "ece",
    "cal_gap", "accuracy",
]
NEEDS_BOTH_SEL = {"auroc", "auprc", "bss", "balanced_accuracy", "ppv", "npv", "f1"}
CLASS_COND_SEL = {"sensitivity", "specificity"}
PREV_DEPENDENT = {"auprc", "bss", "ppv", "npv", "f1", "brier"}
SEL_COVERAGES = (0.9, 0.8, 0.7)
N_RANDOM = 1000
MIN_CLASS_SEL = 3
Z_MDE = norm.ppf(0.975) + norm.ppf(0.80)   # two-sided 95% / 80% power, project convention


def expected_calibration_error(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins - 1)
    n = len(y)
    counts = np.bincount(idx, minlength=n_bins).astype(float)
    sum_y = np.bincount(idx, weights=y, minlength=n_bins)
    sum_p = np.bincount(idx, weights=p, minlength=n_bins)
    with np.errstate(divide="ignore", invalid="ignore"):
        gap = np.abs(sum_y / counts - sum_p / counts)
    gap = np.nan_to_num(gap, nan=0.0)
    return float(np.sum(counts / n * gap))


def binary_metrics(y, p, thr):
    y = np.asarray(y, int); p = np.asarray(p, float); yhat = (p >= thr).astype(int)
    n = len(y)
    out = {"n": n, "prevalence": float(y.mean()) if n else np.nan}
    n_pos = int(y.sum()); n_neg = n - n_pos
    if n_pos > 0 and n_neg > 0:
        ranks = rankdata(p)
        out["auroc"] = float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
        out["auprc"] = average_precision_score(y, p)
    else:
        out["auroc"] = np.nan
        out["auprc"] = np.nan
    tp = int(np.sum((yhat == 1) & (y == 1))); fp = int(np.sum((yhat == 1) & (y == 0)))
    fn = int(np.sum((yhat == 0) & (y == 1))); tn = int(np.sum((yhat == 0) & (y == 0)))
    out["accuracy"] = (tp + tn) / max(1, tp + tn + fp + fn)
    sens = tp / max(1, tp + fn); spec = tn / max(1, tn + fp)
    out["sensitivity"] = sens; out["specificity"] = spec
    out["balanced_accuracy"] = (sens + spec) / 2 if (n_pos > 0 and n_neg > 0) else np.nan
    out["ppv"] = tp / max(1, tp + fp); out["npv"] = tn / max(1, tn + fn)
    out["f1"] = (2 * tp / max(1, 2 * tp + fp + fn)) if (tp + fp + fn) > 0 else 0.0
    out["brier"] = float(np.mean((p - y) ** 2))
    out["ece"] = expected_calibration_error(y, p)
    return out


def bh_fdr(p):
    p = np.asarray(p, float); n = len(p); order = np.argsort(p)
    q = p[order] * n / (np.arange(1, n + 1))
    q = np.clip(np.minimum.accumulate(q[::-1])[::-1], 0, 1)
    out = np.empty(n); out[order] = q
    return out


def suite_on(y, p, thr):
    y = np.asarray(y, int)
    p = np.asarray(p, float)
    m = binary_metrics(y, p, thr)
    prevalence_variance = y.mean() * (1 - y.mean())
    m["bss"] = (1 - np.mean((p - y) ** 2) / prevalence_variance
                if prevalence_variance > 1e-6 else np.nan)
    m["cal_gap"] = float(p.mean() - y.mean())
    return m


def defined_sel(y):
    y = np.asarray(y, int)
    n1 = int((y == 1).sum()); n0 = int((y == 0).sum())
    out = []
    for metric in SEL_SUITE:
        if metric in NEEDS_BOTH_SEL and min(n0, n1) < MIN_CLASS_SEL:
            continue
        if metric == "sensitivity" and n1 < MIN_CLASS_SEL:
            continue
        if metric == "specificity" and n0 < MIN_CLASS_SEL:
            continue
        out.append(metric)
    return out


def suite_masked(y, p, thr, keep):
    y = np.asarray(y, int); p = np.asarray(p, float); keep = np.asarray(keep, bool)
    if keep.sum() == 0:
        return {metric: np.nan for metric in SEL_SUITE}, np.nan
    yk = y[keep]; pk = p[keep]
    defined = set(defined_sel(yk))
    metrics = suite_on(yk, pk, thr)
    return {metric: metrics[metric] if metric in defined else np.nan
            for metric in SEL_SUITE}, float(yk.mean())


def perm_p_two_sided(observed, null_values):
    null_values = np.asarray(null_values, float)
    n = len(null_values)
    if n == 0 or not np.isfinite(observed):
        return np.nan
    p_hi = (np.sum(null_values >= observed) + 1) / (n + 1)
    p_lo = (np.sum(null_values <= observed) + 1) / (n + 1)
    return float(min(1.0, 2 * min(p_hi, p_lo)))


def random_deferral(y, p, thr, n_keep, n_rep=N_RANDOM, seed=0):
    y = np.asarray(y, int); p = np.asarray(p, float)
    n = len(y)
    draws = {metric: np.full(n_rep, np.nan, dtype=float) for metric in SEL_SUITE}
    prevs = np.full(n_rep, np.nan, dtype=float)
    if n_keep <= 0 or n_keep > n:
        return draws, prevs
    rng = np.random.default_rng(seed)
    for i in range(n_rep):
        keep = np.zeros(n, dtype=bool)
        keep[rng.choice(n, n_keep, replace=False)] = True
        values, prevs[i] = suite_masked(y, p, thr, keep)
        for metric in SEL_SUITE:
            draws[metric][i] = values[metric]
    return draws, prevs


def paired_bootstrap_vs_full(y, p, keep, thr, n_boot=N_RANDOM, seed=0):
    """Paired bootstrap: does the FIXED, already-calibration-locked retained
    set differ significantly from that same resample's own full-cohort
    (no-deferral) metrics? One shared resample per draw for both quantities,
    generalising this project's paired_bootstrap_auroc pattern (primary
    codel notebook, Part D) to the full SEL_SUITE panel. Reuses the
    original, already-fixed keep/abstain label per patient (keep[bi] on a
    resample equals recomputing s[bi] <= cutoff, since resampling only
    reindexes)."""
    y = np.asarray(y, int); p = np.asarray(p, float); keep = np.asarray(keep, bool)
    n = len(y)
    rng = np.random.default_rng(seed)
    diffs = {metric: [] for metric in SEL_SUITE}
    attempts = 0
    n_target = n_boot
    while len(diffs["accuracy"]) < n_target and attempts < n_boot * 30:
        attempts += 1
        bi = rng.integers(0, n, n)
        yb, pb, kb = y[bi], p[bi], keep[bi]
        if len(np.unique(yb)) < 2:
            continue
        full_b = suite_on(yb, pb, thr)
        ret_b, _ = suite_masked(yb, pb, thr, kb)
        for metric in SEL_SUITE:
            fb, rb = full_b.get(metric, np.nan), ret_b.get(metric, np.nan)
            if np.isfinite(fb) and np.isfinite(rb):
                diffs[metric].append(rb - fb)
    out = {}
    for metric in SEL_SUITE:
        d = np.asarray(diffs[metric], float)
        if len(d) < 50:
            out[metric] = {"d_full": np.nan, "d_full_lo": np.nan, "d_full_hi": np.nan,
                            "d_full_mde": np.nan, "p_full_raw": np.nan, "n_boot_full": len(d)}
            continue
        lo, hi = np.percentile(d, [2.5, 97.5])
        p_two = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
        se = float(d.std(ddof=1))
        out[metric] = {"d_full": float(d.mean()), "d_full_lo": float(lo), "d_full_hi": float(hi),
                        "d_full_mde": Z_MDE * se, "p_full_raw": p_two, "n_boot_full": len(d)}
    return out


def fit_calibration_cutoffs(cal, signal, coverages=SEL_COVERAGES):
    s_cal = cal[signal].astype(float).to_numpy()
    if not np.isfinite(s_cal).all():
        raise ValueError(f"Calibration signal {signal!r} contains non-finite values.")
    if len(s_cal) == 0:
        raise ValueError(f"No calibration observations available for {signal!r}.")
    return {float(tc): float(np.quantile(s_cal, tc)) for tc in coverages}


def selective_table_locked(cal, ev, prob_col, thr, signal, coverages=SEL_COVERAGES, seed=0):
    y = ev["label"].astype(int).to_numpy()
    p = ev[prob_col].astype(float).to_numpy()
    s_ev = ev[signal].astype(float).to_numpy()
    if not np.isfinite(s_ev).all():
        raise ValueError(f"Evaluation signal {signal!r} contains non-finite values.")
    full = suite_on(y, p, thr)
    cutoffs = fit_calibration_cutoffs(cal=cal, signal=signal, coverages=coverages)
    rows = []
    for target_cov in coverages:
        target_cov = float(target_cov)
        cutoff = cutoffs[target_cov]
        keep = s_ev <= cutoff
        n_keep = int(keep.sum())
        achieved_cov = float(keep.mean())
        values, retained_prev = suite_masked(y=y, p=p, thr=thr, keep=keep)
        random_draws, random_prevs = random_deferral(y=y, p=p, thr=thr, n_keep=n_keep, n_rep=N_RANDOM, seed=seed)
        full_test = paired_bootstrap_vs_full(y=y, p=p, keep=keep, thr=thr, n_boot=N_RANDOM, seed=seed)
        for metric in SEL_SUITE:
            random_values = random_draws[metric]
            random_values = random_values[np.isfinite(random_values)]
            observed = values[metric]
            has_comparison = np.isfinite(observed) and len(random_values) > 0
            ft = full_test[metric]
            rows.append({
                "signal": signal, "target_coverage": target_cov, "achieved_coverage": achieved_cov,
                "coverage_shift": achieved_cov - target_cov, "calibration_cutoff": cutoff, "metric": metric,
                "full": full[metric], "retained": observed,
                "delta_vs_full": observed - full[metric] if np.isfinite(observed) else np.nan,
                "d_full": ft["d_full"], "d_full_lo": ft["d_full_lo"], "d_full_hi": ft["d_full_hi"],
                "d_full_mde": ft["d_full_mde"], "p_full_raw": ft["p_full_raw"],
                "random_mean": float(random_values.mean()) if len(random_values) else np.nan,
                "random_lo": float(np.percentile(random_values, 2.5)) if len(random_values) else np.nan,
                "random_hi": float(np.percentile(random_values, 97.5)) if len(random_values) else np.nan,
                "delta_vs_random": observed - float(random_values.mean()) if has_comparison else np.nan,
                "pct_in_random": float((random_values < observed).mean()) if has_comparison else np.nan,
                "p_raw": perm_p_two_sided(observed, random_values) if has_comparison else np.nan,
                "retained_prev": retained_prev,
                "random_prev": float(np.nanmean(random_prevs)) if np.isfinite(random_prevs).any() else np.nan,
                "full_prev": float(y.mean()), "n_retained": n_keep, "n_evaluation": len(y),
            })
    return pd.DataFrame(rows)


print("Calibration-locked deferral machinery loaded (ported from main bench cell 67, "
      "extended with the vs-full-baseline paired bootstrap test, mirroring cell 78's own fix).")


Calibration-locked deferral machinery loaded (ported from main bench cell 67, extended with the vs-full-baseline paired bootstrap test, mirroring cell 78's own fix).


In [ ]:
SELECTIVE_IPD = {}
picks_by_tier = {}

for tier_name, tier_mask in [("confident", master["atrx_concordant"] == True),
                              ("full", pd.Series(True, index=master.index))]:
    sub = master[tier_mask]
    EV = common_cohort(sub, sig_cols_ipd, ["label", "prob_mutant"])
    CA = common_cohort(codel_tcga_cal, sig_cols_ipd, ["label", "prob_mutant"])

    idx = boot_indices(len(EV))
    tab, draws, err = evaluate_signals(EV, "prob_mutant", LOCKED_THR, fams_ipd, idx)
    picks = {family: tab.query("family == @family").iloc[0]["signal"]
             for family in fams_ipd}
    picks_by_tier[tier_name] = picks

    outputs = []
    for family, signal in picks.items():
        out = selective_table_locked(
            cal=CA, ev=EV, prob_col="prob_mutant", thr=LOCKED_THR,
            signal=signal, coverages=SEL_COVERAGES, seed=0,
        ).assign(family=family, predictor="ensemble", tier=tier_name)
        outputs.append(out)
    SELECTIVE_IPD[tier_name] = pd.concat(outputs, ignore_index=True)
    print(f"{tier_name}: n_eval={len(EV)}, n_cal={len(CA)}")
    print(f"  picks: {picks}")

# --- BH-FDR within each metric, across every family x coverage x tier row ---
ALL_SELECTIVE_IPD = pd.concat(SELECTIVE_IPD.values(), ignore_index=True)
ALL_SELECTIVE_IPD["q_fdr"] = np.nan
for _metric, _idx in ALL_SELECTIVE_IPD.groupby("metric").groups.items():
    _sub = ALL_SELECTIVE_IPD.loc[_idx]
    _ok = _sub["p_raw"].notna()
    ALL_SELECTIVE_IPD.loc[_sub.index[_ok], "q_fdr"] = bh_fdr(_sub.loc[_ok, "p_raw"].to_numpy())
ALL_SELECTIVE_IPD["sig_fdr"] = ALL_SELECTIVE_IPD["q_fdr"] < 0.05

# --- same correction, second test: vs full-cohort baseline ---
ALL_SELECTIVE_IPD["q_fdr_full"] = np.nan
for _metric, _idx in ALL_SELECTIVE_IPD.groupby("metric").groups.items():
    _sub = ALL_SELECTIVE_IPD.loc[_idx]
    _ok = _sub["p_full_raw"].notna()
    ALL_SELECTIVE_IPD.loc[_sub.index[_ok], "q_fdr_full"] = bh_fdr(_sub.loc[_ok, "p_full_raw"].to_numpy())
ALL_SELECTIVE_IPD["sig_fdr_full"] = ALL_SELECTIVE_IPD["q_fdr_full"] < 0.05

ALL_SELECTIVE_IPD.to_csv(OUT_DIR / "selective_metrics_cal_locked_all_fdr.csv", index=False)
print(f"\nSaved {OUT_DIR / 'selective_metrics_cal_locked_all_fdr.csv'} ({len(ALL_SELECTIVE_IPD)} rows)")

# --- AUROC summary, both tiers, with both significance flags ---
summary = ALL_SELECTIVE_IPD.query("metric == 'auroc'")[
    ["tier", "family", "signal", "target_coverage", "achieved_coverage", "full", "retained",
     "delta_vs_full", "p_full_raw", "q_fdr_full", "sig_fdr_full",
     "random_mean", "delta_vs_random", "p_raw", "q_fdr", "sig_fdr",
     "n_retained", "n_evaluation"]
].sort_values(["tier", "target_coverage", "family"])
display(summary.round(4))

n_sig_auroc = int(summary["sig_fdr"].sum())
n_sig_auroc_full = int(summary["sig_fdr_full"].sum())
n_total_auroc = len(summary)
print(f"\nAUROC rows surviving BH-FDR q<0.05 vs random-deferral: {n_sig_auroc}/{n_total_auroc}")
print(f"AUROC rows surviving BH-FDR q<0.05 vs full-cohort baseline: {n_sig_auroc_full}/{n_total_auroc}")

n_sig_all = int(ALL_SELECTIVE_IPD["sig_fdr"].sum())
n_sig_all_full = int(ALL_SELECTIVE_IPD["sig_fdr_full"].sum())
n_total_all = ALL_SELECTIVE_IPD["p_raw"].notna().sum()
n_total_all_full = ALL_SELECTIVE_IPD["p_full_raw"].notna().sum()
print(f"All-metric rows surviving BH-FDR q<0.05 vs random-deferral: {n_sig_all}/{n_total_all}")
print(f"All-metric rows surviving BH-FDR q<0.05 vs full-cohort baseline: {n_sig_all_full}/{n_total_all_full}")


confident: n_eval=127, n_cal=66
  picks: {'predictive': 'aleatoric', 'epistemic': 'mean_within_encoder_mi', 'ood': 'ood_attention_uni2', 'mc_dropout': 'mc_aleatoric', 'deep_ensemble': 'de_aleatoric', 'laplace': 'laplace_aleatoric'}
full: n_eval=138, n_cal=66
  picks: {'predictive': 'aleatoric', 'epistemic': 'mean_within_encoder_mi', 'ood': 'ood_attention_uni2', 'mc_dropout': 'mc_aleatoric', 'deep_ensemble': 'de_aleatoric', 'laplace': 'laplace_aleatoric'}

Saved /cs/student/project_msc/2025/aibh/mpapageo/outputs/codel_ipd_brain_extension/selective_metrics_cal_locked_all_fdr.csv (468 rows)


,tier,family,signal,target_coverage,achieved_coverage,full,retained,delta_vs_full,p_full_raw,q_fdr_full,sig_fdr_full,random_mean,delta_vs_random,p_raw,q_fdr,sig_fdr,n_retained,n_evaluation
182,confident,deep_ensemble,de_aleatoric,0.7,0.4094,0.9225,0.9670,0.0445,0.136,0.1840,False,0.9240,0.0430,0.1099,0.1413,False,52,127
65,confident,epistemic,mean_within_encoder_mi,0.7,0.5827,0.9225,0.9632,0.0407,0.010,0.0900,False,0.9241,0.0391,0.0340,0.1028,False,74,127
221,confident,laplace,laplace_aleatoric,0.7,0.4252,0.9225,0.9682,0.0457,0.120,0.1840,False,0.9221,0.0461,0.0759,0.1139,False,54,127
143,confident,mc_dropout,mc_aleatoric,0.7,0.4252,0.9225,0.9655,0.0430,0.176,0.2112,False,0.9221,0.0434,0.1079,0.1413,False,54,127
104,confident,ood,ood_attention_uni2,0.7,0.9055,0.9225,0.9159,-0.0066,0.296,0.3437,False,0.9226,-0.0067,0.3856,0.4338,False,115,127
26,confident,predictive,aleatoric,0.7,0.4252,0.9225,0.9655,0.0430,0.176,0.2112,False,0.9221,0.0434,0.1079,0.1413,False,54,127
169,confident,deep_ensemble,de_aleatoric,0.8,0.4646,0.9225,0.9675,0.0450,0.126,0.1840,False,0.9235,0.0439,0.0599,0.1028,False,59,127
52,confident,epistemic,mean_within_encoder_mi,0.8,0.8189,0.9225,0.9472,0.0247,0.006,0.0900,False,0.9228,0.0244,0.0380,0.1028,False,104,127
208,confident,laplace,laplace_aleatoric,0.8,0.4488,0.9225,0.9687,0.0462,0.112,0.1840,False,0.9225,0.0461,0.0559,0.1028,False,57,127
130,confident,mc_dropout,mc_aleatoric,0.8,0.4724,0.9225,0.9669,0.0444,0.134,0.1840,False,0.9228,0.0441,0.0559,0.1028,False,60,127



AUROC rows surviving BH-FDR q<0.05 vs random-deferral: 0/36
AUROC rows surviving BH-FDR q<0.05 vs full-cohort baseline: 0/36
All-metric rows surviving BH-FDR q<0.05 vs random-deferral: 244/468
All-metric rows surviving BH-FDR q<0.05 vs full-cohort baseline: 265/468


## Total-risk 2-stage pipeline (secondary, doubly-caveated)

Mirrors the primary codel notebook's own Phase 3. Every accuracy number
above is measured on the codel-eligible cohort, patients with
(proxy-)known IDH-mutant status, which implicitly assumes Stage 1 (IDH
classification) is 100% accurate, since it never runs Stage 1 at all.
This asks the same complementary question as the TCGA notebook: what is
the true end-to-end automated accuracy of the full two-stage pipeline on
the real, undifferentiated IPD Brain population, compared against the
codel-eligible cohort's own reported baseline accuracy?

Two caveats stack here, on top of everything already flagged for this
notebook's codel arm generally (ATRX/Subtype proxy label, not a real
1p/19q assay):

1. Both stages' cutoffs and thresholds are TCGA-transferred, not locally
   fit. IPD Brain has no calibration subset of its own in this pipeline
   (everything here is external evaluation by construction), so there's
   no local-refitting comparison available the way there is for TCGA.
2. 23 of IPD's 161 true-IDH-mutant patients have no codel data at all
   (14.3% of the true-mutant population), larger than TCGA's equivalent
   gap (2/418, 0.5%). Patients Stage 1 would route to codeletion testing
   who happen to be among these 23 are excluded from the automated
   accuracy calculation, the same "no data available" treatment as the
   primary notebook.

Given both stacked caveats, this section is explicitly secondary and
exploratory: directional support for the TCGA finding if the pattern
replicates, not independent confirmation on its own.

In [10]:
# ============================================================
# Total-risk 2-stage pipeline, IPD Brain -- TCGA-transferred cutoffs at
# both stages (no local refitting available for this cohort).
# ============================================================
THR1 = 0.441619   # main IDH bench's locked ensemble threshold (Youden J, TCGA cal)

idh_ipd_master = pd.read_csv(BASE / "outputs/ipd_brain_baseline_bench/master_patient_table.csv")
idh_ipd_master["patient"] = idh_ipd_master["patient"].astype(str)

idh_cutoffs = pd.read_csv(BASE / "outputs/selective_prediction_bench/selective_metrics_cal_locked_all_fdr.csv")
idh_pred_cutoffs = (idh_cutoffs[(idh_cutoffs.predictor == "ensemble") & (idh_cutoffs.family == "predictive")]
                     .drop_duplicates(["target_coverage"])[["target_coverage", "signal", "calibration_cutoff"]]
                     .sort_values("target_coverage", ascending=False))
print("Stage-1 (IDH) TCGA-locked cutoffs, transferred unchanged:")
display(idh_pred_cutoffs)

# Stage-2 cutoffs: TCGA-transferred, fit on codel_tcga_cal (already loaded above,
# same n=66 codel calibration set used for LOCKED_THR).
stage2_cutoffs = fit_calibration_cutoffs(cal=codel_tcga_cal, signal="ensemble_entropy", coverages=SEL_COVERAGES)
print("\nStage-2 (codel) TCGA-locked cutoffs, transferred unchanged:")
for cov, cutoff in stage2_cutoffs.items():
    print(f"  target_coverage={cov:.0%}  cutoff={cutoff:.6f}")

THR2 = LOCKED_THR  # already computed above, TCGA-locked, Youden J on codel_tcga_cal

m = idh_ipd_master[["patient", "label", "prob_mutant", "ensemble_entropy"]].rename(
    columns={"label": "y1", "prob_mutant": "p1", "ensemble_entropy": "h1"})


def run_pipeline(df, c1, c2, thr1, thr2, y2col, p2col, h2col):
    """Returns (n_automated, n_correct, n_total, breakdown) -- breakdown is a
    dict of per-stage counts, same convention as the primary codel notebook's
    own run_pipeline, added here for the combined TCGA/IPD stage-tradeoff
    table. Always computed (cheap); the bootstrap loop below ignores it."""
    defer1 = df["h1"] > c1
    call1_mutant = df["p1"] >= thr1
    retained1 = ~defer1

    outcome = np.full(len(df), "", dtype=object)
    outcome[defer1.to_numpy()] = "deferred1"

    ends_wt = (retained1 & ~call1_mutant).to_numpy()
    y1 = df["y1"].to_numpy()
    outcome[ends_wt & (y1 == 0)] = "correct"
    outcome[ends_wt & (y1 == 1)] = "wrong"

    fp_wrong_pathway = (retained1 & call1_mutant & (df["y1"] == 0)).to_numpy()
    outcome[fp_wrong_pathway] = "wrong"

    proceeds = (retained1 & call1_mutant & (df["y1"] == 1)).to_numpy()
    has_codel = df[y2col].notna().to_numpy()
    no_codel = proceeds & ~has_codel
    outcome[no_codel] = "excluded"

    proceeds_ok = proceeds & has_codel
    h2 = df[h2col].to_numpy()
    defer2 = proceeds_ok & (h2 > c2)
    outcome[defer2] = "deferred2"

    retained2 = proceeds_ok & ~defer2
    p2 = df[p2col].to_numpy()
    y2 = df[y2col].to_numpy()
    call2_pos = p2 >= thr2
    correct2 = retained2 & (call2_pos == y2)
    wrong2 = retained2 & (call2_pos != y2)
    outcome[correct2] = "correct"
    outcome[wrong2] = "wrong"

    assert (outcome != "").all(), "unhandled row in pipeline outcome"
    n_total = len(df)
    n_automated = int((outcome == "correct").sum() + (outcome == "wrong").sum())
    n_correct = int((outcome == "correct").sum())

    breakdown = {
        "n_total": n_total,
        "stage1_deferred_n": int(defer1.sum()),
        "stage1_retained_n": int(retained1.sum()),
        "stage1_ends_wt_n": int(ends_wt.sum()),
        "stage1_ends_wt_correct_n": int((ends_wt & (y1 == 0)).sum()),
        "stage1_fp_wrong_pathway_n": int(fp_wrong_pathway.sum()),
        "proceeds_n": int(proceeds.sum()),
        "no_codel_excluded_n": int(no_codel.sum()),
        "proceeds_ok_n": int(proceeds_ok.sum()),
        "stage2_deferred_n": int(defer2.sum()),
        "stage2_retained_n": int(retained2.sum()),
        "stage2_correct_n": int(correct2.sum()),
        "stage2_wrong_n": int(wrong2.sum()),
    }
    return n_automated, n_correct, n_total, breakdown


N_BOOT_PIPELINE = 2000
rng = np.random.default_rng(0)

all_results = []
breakdown_rows = []
for tier_name, tier_mask_fn in [
    ("ATRX-confident (headline)", lambda d: d["atrx_concordant"] == True),
    ("Full 138 (incl. discordant)", lambda d: pd.Series(True, index=d.index)),
]:
    codel_tier = master[tier_mask_fn(master)]
    md_tier = m.merge(
        codel_tier[["patient", "label", "prob_mutant", "ensemble_entropy"]].rename(
            columns={"label": "y2", "prob_mutant": "p2", "ensemble_entropy": "h2"}),
        on="patient", how="left")

    # full-set codeletion accuracy (oracle Stage-1), this tier's own baseline
    call2_full = (codel_tier["prob_mutant"] >= THR2).astype(int)
    full_set_acc = (call2_full == codel_tier["label"]).mean()
    oracle_merge = codel_tier.merge(idh_ipd_master[["patient", "prob_mutant"]].rename(columns={"prob_mutant": "p1"}),
                                     on="patient", how="left")
    stage1_would_miss = (oracle_merge["p1"] < THR1).mean()

    print(f"\n{'='*90}\n{tier_name}: n={len(codel_tier)}\n{'='*90}")
    print(f"Full-set codeletion accuracy (oracle Stage-1): {full_set_acc:.4f}")
    print(f"Stage-1 would itself misclassify {stage1_would_miss:.1%} of this 'known mutant' population")

    n = len(md_tier)
    for cov in SEL_COVERAGES:
        c1 = float(idh_pred_cutoffs.query("target_coverage == @cov")["calibration_cutoff"].iloc[0])
        c2 = stage2_cutoffs[cov]

        n_auto, n_correct, n_total, bd = run_pipeline(md_tier, c1, c2, THR1, THR2, "y2", "p2", "h2")
        point_acc = n_correct / n_auto
        point_cov = n_auto / n_total

        boot_acc, boot_cov = [], []
        for _ in range(N_BOOT_PIPELINE):
            bi = rng.integers(0, n, size=n)
            bdf = md_tier.iloc[bi].reset_index(drop=True)
            ba, bc, bt, _ = run_pipeline(bdf, c1, c2, THR1, THR2, "y2", "p2", "h2")
            if ba > 0:
                boot_acc.append(bc / ba)
                boot_cov.append(ba / bt)

        acc_lo, acc_hi = np.percentile(boot_acc, [2.5, 97.5])
        cov_lo, cov_hi = np.percentile(boot_cov, [2.5, 97.5])

        all_results.append({
            "tier": tier_name, "target_coverage": cov, "n_automated": n_auto, "n_total": n_total,
            "automation_rate": point_cov, "automation_rate_lo": cov_lo, "automation_rate_hi": cov_hi,
            "pipeline_accuracy": point_acc, "pipeline_accuracy_lo": acc_lo, "pipeline_accuracy_hi": acc_hi,
            "full_set_accuracy": full_set_acc, "stage1_error_on_known_mutant_pop": stage1_would_miss,
        })

        # --- per-stage breakdown, point estimate only (descriptive, not
        # separately bootstrapped -- pipeline_accuracy above carries the CI) ---
        breakdown_rows.append({
            "tier": tier_name, "target_coverage": cov,
            "stage1_deferred_pct": bd["stage1_deferred_n"] / bd["n_total"],
            "stage1_deferred_n": bd["stage1_deferred_n"],
            "stage1_retained_n": bd["stage1_retained_n"],
            "stage1_terminal_wt_n": bd["stage1_ends_wt_n"],
            "stage1_terminal_wt_accuracy": bd["stage1_ends_wt_correct_n"] / max(1, bd["stage1_ends_wt_n"]),
            "stage1_fp_wrong_pathway_n": bd["stage1_fp_wrong_pathway_n"],
            "proceeds_to_stage2_n": bd["proceeds_n"],
            "excluded_no_codel_data_n": bd["no_codel_excluded_n"],
            "stage2_deferred_pct_of_proceeds": bd["stage2_deferred_n"] / max(1, bd["proceeds_ok_n"]),
            "stage2_deferred_n": bd["stage2_deferred_n"],
            "stage2_retained_n": bd["stage2_retained_n"],
            "stage2_accuracy": bd["stage2_correct_n"] / max(1, bd["stage2_retained_n"]),
        })

ipd_pipeline_df = pd.DataFrame(all_results)
ipd_pipeline_df.to_csv(OUT_DIR / "two_stage_total_risk_pipeline.csv", index=False)
print(f"\nSaved {OUT_DIR / 'two_stage_total_risk_pipeline.csv'}")
display(ipd_pipeline_df.round(4))

for _, r in ipd_pipeline_df.iterrows():
    overlap = not (r.pipeline_accuracy_lo > r.full_set_accuracy or r.pipeline_accuracy_hi < r.full_set_accuracy)
    print(f"[{r.tier}] cov={r.target_coverage:.0%}: pipeline {r.pipeline_accuracy:.4f} "
          f"[{r.pipeline_accuracy_lo:.4f},{r.pipeline_accuracy_hi:.4f}] vs full-set {r.full_set_accuracy:.4f} "
          f"-- CIs {'OVERLAP' if overlap else 'DO NOT OVERLAP (pipeline higher)'}")

ipd_breakdown_df = pd.DataFrame(breakdown_rows)
ipd_breakdown_df.to_csv(OUT_DIR / "two_stage_pipeline_stage_breakdown.csv", index=False)
print(f"\n{'='*90}")
print("Per-stage breakdown (point estimates, descriptive), both tiers")
print(f"{'='*90}")
display(ipd_breakdown_df.round(4))


Stage-1 (IDH) TCGA-locked cutoffs, transferred unchanged:


,target_coverage,signal,calibration_cutoff
1092,0.9,ensemble_entropy,0.628037
1105,0.8,ensemble_entropy,0.552999
1118,0.7,ensemble_entropy,0.408982



Stage-2 (codel) TCGA-locked cutoffs, transferred unchanged:
  target_coverage=90%  cutoff=0.620323
  target_coverage=80%  cutoff=0.468195
  target_coverage=70%  cutoff=0.380856

ATRX-confident (headline): n=127
Full-set codeletion accuracy (oracle Stage-1): 0.6850
Stage-1 would itself misclassify 13.4% of this 'known mutant' population

Full 138 (incl. discordant): n=138
Full-set codeletion accuracy (oracle Stage-1): 0.6522
Stage-1 would itself misclassify 15.9% of this 'known mutant' population

Saved /cs/student/project_msc/2025/aibh/mpapageo/outputs/codel_ipd_brain_extension/two_stage_total_risk_pipeline.csv


,tier,target_coverage,n_automated,n_total,automation_rate,automation_rate_lo,automation_rate_hi,pipeline_accuracy,pipeline_accuracy_lo,pipeline_accuracy_hi,full_set_accuracy,stage1_error_on_known_mutant_pop
0,ATRX-confident (headline),0.9,244,320,0.7625,0.7156,0.8062,0.8156,0.7683,0.8640,0.6850,0.1339
1,ATRX-confident (headline),0.8,212,320,0.6625,0.6094,0.7156,0.8632,0.8160,0.9082,0.6850,0.1339
2,ATRX-confident (headline),0.7,179,320,0.5594,0.5031,0.6125,0.8771,0.8281,0.9255,0.6850,0.1339
3,Full 138 (incl. discordant),0.9,245,320,0.7656,0.7188,0.8125,0.8163,0.7667,0.8618,0.6522,0.1594
4,Full 138 (incl. discordant),0.8,212,320,0.6625,0.6125,0.7156,0.8632,0.8173,0.9078,0.6522,0.1594
5,Full 138 (incl. discordant),0.7,179,320,0.5594,0.5062,0.6125,0.8771,0.8274,0.9240,0.6522,0.1594


[ATRX-confident (headline)] cov=90%: pipeline 0.8156 [0.7683,0.8640] vs full-set 0.6850 -- CIs DO NOT OVERLAP (pipeline higher)
[ATRX-confident (headline)] cov=80%: pipeline 0.8632 [0.8160,0.9082] vs full-set 0.6850 -- CIs DO NOT OVERLAP (pipeline higher)
[ATRX-confident (headline)] cov=70%: pipeline 0.8771 [0.8281,0.9255] vs full-set 0.6850 -- CIs DO NOT OVERLAP (pipeline higher)
[Full 138 (incl. discordant)] cov=90%: pipeline 0.8163 [0.7667,0.8618] vs full-set 0.6522 -- CIs DO NOT OVERLAP (pipeline higher)
[Full 138 (incl. discordant)] cov=80%: pipeline 0.8632 [0.8173,0.9078] vs full-set 0.6522 -- CIs DO NOT OVERLAP (pipeline higher)
[Full 138 (incl. discordant)] cov=70%: pipeline 0.8771 [0.8274,0.9240] vs full-set 0.6522 -- CIs DO NOT OVERLAP (pipeline higher)

Per-stage breakdown (point estimates, descriptive), both tiers


,tier,target_coverage,stage1_deferred_pct,stage1_deferred_n,stage1_retained_n,stage1_terminal_wt_n,stage1_terminal_wt_accuracy,stage1_fp_wrong_pathway_n,proceeds_to_stage2_n,excluded_no_codel_data_n,stage2_deferred_pct_of_proceeds,stage2_deferred_n,stage2_retained_n,stage2_accuracy
0,ATRX-confident (headline),0.9,0.1531,49,271,173,0.8266,5,93,5,0.2500,22,66,0.8485
1,ATRX-confident (headline),0.8,0.2438,78,242,164,0.8598,5,73,2,0.3944,28,43,0.9767
2,ATRX-confident (headline),0.7,0.3562,114,206,145,0.8759,3,58,0,0.4655,27,31,0.9677
3,Full 138 (incl. discordant),0.9,0.1531,49,271,173,0.8266,5,93,2,0.2637,24,67,0.8507
4,Full 138 (incl. discordant),0.8,0.2438,78,242,164,0.8598,5,73,0,0.4110,30,43,0.9767
5,Full 138 (incl. discordant),0.7,0.3562,114,206,145,0.8759,3,58,0,0.4655,27,31,0.9677


In [ ]:
_conf = master[master["atrx_concordant"] == True].reset_index(drop=True)
_y = _conf["label"].to_numpy(int)
_p = _conf["prob_mutant"].to_numpy(float)
_h = _conf["ensemble_entropy"].to_numpy(float)
_call = (_p >= THR2).astype(int)
_correct = (_call == _y).astype(int)
_order = np.argsort(_h)  # ascending: most confident (lowest entropy) first

_bd_conf = ipd_breakdown_df[ipd_breakdown_df.tier == "ATRX-confident (headline)"].set_index("target_coverage")
_pipe_conf = ipd_pipeline_df[ipd_pipeline_df.tier == "ATRX-confident (headline)"].set_index("target_coverage")

_rng2 = np.random.default_rng(0)
N_BOOT_ORACLE = 5000
n_total_conf = len(_conf)

print(f"{'target':>8} {'n match':>8} {'oracle+defer':>22} {'real Stage-2':>13} {'real pipeline':>14}")
_rows = []
for cov in SEL_COVERAGES:
    n_match = int(_bd_conf.loc[cov, "stage2_retained_n"])
    top_idx = _order[:n_match]
    point_acc = _correct[top_idx].mean()

    accs = []
    for _ in range(N_BOOT_ORACLE):
        bi = _rng2.integers(0, n_total_conf, size=n_total_conf)
        bh, bcorrect = _h[bi], _correct[bi]
        bord = np.argsort(bh)[:n_match]
        accs.append(bcorrect[bord].mean())
    lo, hi = np.percentile(accs, [2.5, 97.5])

    real_stage2_acc = _bd_conf.loc[cov, "stage2_accuracy"]
    real_pipeline_acc = _pipe_conf.loc[cov, "pipeline_accuracy"]
    print(f"{cov:>7.0%} {n_match:>8d} {point_acc:>8.4f} [{lo:.3f},{hi:.3f}]"
          f" {real_stage2_acc:>13.4f} {real_pipeline_acc:>14.4f}")
    _rows.append({
        "target_coverage": cov, "n_match": n_match,
        "oracle_informed_deferral_acc": point_acc,
        "oracle_informed_deferral_acc_lo": lo, "oracle_informed_deferral_acc_hi": hi,
        "real_stage2_only_acc": real_stage2_acc, "real_pipeline_acc": real_pipeline_acc,
    })

_oracle_vs_real_df = pd.DataFrame(_rows)
_oracle_vs_real_df.to_csv(OUT_DIR / "oracle_vs_real_stage1_matched_n.csv", index=False)
print(f"\nSaved {OUT_DIR / 'oracle_vs_real_stage1_matched_n.csv'}")

  target  n match           oracle+defer  real Stage-2  real pipeline
    90%       66   0.9848 [0.894,1.000]        0.8485         0.8156
    80%       43   0.9767 [0.930,1.000]        0.9767         0.8632
    70%       31   0.9677 [0.903,1.000]        0.9677         0.8771

Saved /cs/student/project_msc/2025/aibh/mpapageo/outputs/codel_ipd_brain_extension/oracle_vs_real_stage1_matched_n.csv


In [ ]:
_rng4 = np.random.default_rng(0)
N_BOOT_BASELINE = 5000
_n_base = len(_conf)
_y_base = _conf["label"].to_numpy(int)
_call_base = (_conf["prob_mutant"].to_numpy(float) >= THR2).astype(int)
_correct_base = (_call_base == _y_base).astype(int)

_base_accs = np.array([_correct_base[_rng4.integers(0, _n_base, size=_n_base)].mean()
                        for _ in range(N_BOOT_BASELINE)])
_base_lo, _base_hi = np.percentile(_base_accs, [2.5, 97.5])
print(f"IPD Brain (confident) no-deferral baseline: {_correct_base.mean():.4f} [{_base_lo:.4f},{_base_hi:.4f}]  (n={_n_base})")

IPD Brain (confident) no-deferral baseline: 0.6850 [0.6063,0.7638]  (n=127)
